# A quick and simple creation of model using pytorch lightning

This notebook contains a simple and minimalistic introduction into pytorch and pytorch lightning. We will create a simple model and train it.

## Pytorch data loading



In [ ]:
import os

from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torchvision import datasets, transforms
import timm
import torch
import matplotlib.pyplot as plt

from lit_ecology_classifier.data.image_transformation import define_transformation_pipeline

transformations = define_transformation_pipeline(train=True)



# Use 
data_dir = os.path.join("..", "data", "mini_dataset")

# Minidataset to check if it learns something
#full_dataset = torch.utils.data.Subset(full_dataset, range(1200))  

full_dataset = datasets.ImageFolder(root=data_dir, transform=transformations)

# Print some information about the dataset
print("Classes:", full_dataset.classes)
print("Class:", full_dataset.class_to_idx)
print("Gesamtzahl der Bilder:", len(full_dataset))




# Split the dataset into training, validation and test set
train_size = int(0.7 * len(full_dataset))  # 70% Train
val_size = int(0.15 * len(full_dataset))   # 15% Val
test_size = len(full_dataset) - train_size - val_size  # 15% Test

print("Trainingsgröße:", train_size)
print("Validierungsgröße:", val_size)
print("Testgröße:", test_size)

# Random split
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size, test_size])



In [ ]:
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=5)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=5)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Example of a batch images

In [ ]:
single_batch = next(iter(train_loader))
image, label = single_batch



def plot_images(images, labels):
    fig, axes = plt.subplots(4, 4, figsize=(15, 15))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i].permute(1, 2, 0))
        ax.axis('off')
        ax.set_title(labels[i].item())
    plt.show()

plot_images(image, label)

In [ ]:
len_classes = len(full_dataset.classes)
print(len_classes)

# Create a simple model 

We implented only the train and validation steps for the model, since it is a simple introduction example. Feel free to add the test step :)

## Load model from timm

In [ ]:
def get_model(num_classes=2):
    # Load MobileNetV3 Large from timm
    model = timm.create_model(
        'mobilenetv3_large_100.miil_in21k_ft_in1k',
        pretrained=True,
        num_classes=num_classes  # 2 classes: 0 and 1
    )

    # Freeze all layers except the last one
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model
model = get_model(len_classes)

## Simple model with logs 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import lightning as pl
from torchmetrics import Accuracy
import torch.nn.functional as F
from lightning.pytorch.loggers import CSVLogger
from lightning import Trainer

csv_logger = CSVLogger("logs", name="my_experiment")


from lit_ecology_classifier.helpers.modelling_plots import plot_loss_acc


class MinimalisticalisticModel(pl.LightningModule):
    def __init__(self, model, lr=0.001):
        super(MinimalisticalisticModel, self).__init__()
        self.model = model
        self.lr = lr
        self.criterion = nn.CrossEntropyLoss()
        self.train_acc = Accuracy(task="multiclass", num_classes=len_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=len_classes)

    def forward(self, x):
        # Forward pass of the model
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.model(inputs)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        acc = self.train_acc(preds, labels)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.model(inputs)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1) 
        acc = self.val_acc(preds, labels)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def on_validation_end(self):
        return super().on_validation_end()

    def configure_optimizers(self):
        return optim.Adam(self.model.parameters(), lr=self.lr)
    
    def on_fit_end(self) -> None:
        """
        If the model is not using wandb, plot the loss and accuracy curves at the end of training
        and save them in the output folder.
        """
        print(self.logger.log_dir)
        
        plot_loss_acc(self.trainer.logger)
        plt.show(plot_loss_acc)
        return super().on_fit_end()
    
    def on_test_start(self):
        return super().on_test_start()
    
    def test_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.model(inputs)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        acc = self.val_acc(preds, labels)
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return {"loss": loss, "acc": acc}

    def on_test_epoch_end(self, outputs):
        
        avg_loss = torch.stack([x["loss"] for x in outputs]).mean()
        avg_acc = torch.stack([x["acc"] for x in outputs]).mean()
        print("Test loss:", avg_loss)
        print("Test accuracy:", avg_acc)
        return super().on_test_epoch_end()
        
        

## Init a minimalistic model and trainer

Pytorch lightning provides a simple way to define the model and the training loop. You can define the model in a `LightningModule` class. The `trainer` class handles the training and evertything else like devices and parallelization strategies.


In [ ]:
# if you have a cuda enabled device, you can set the test_on to 'gpu'.capitalize
test_on = 'gpu' if torch.cuda.is_available() else 'cpu'

In [ ]:
from lightning.pytorch.callbacks  import ModelCheckpoint,LearningRateMonitor, EarlyStopping

mini_Model = MinimalisticalisticModel(model)

# Define the model checkpoint callback
checkpoint_callback = [ModelCheckpoint(
    monitor="val_loss",        # Which metric to monitor
    mode="min",                # How to interpret the monitor ( loss we want to minimize), accuracy we want to maximize)
    dirpath="checkpoints",     # Save the best model in this folder
    filename="model-{epoch:02d}-{val_loss:.2f}",  # The name of the best model
    save_top_k=3,              # How many models to save   
), 
LearningRateMonitor(logging_interval="step"), # Log the learning rate at every step, also common to log at epoch
EarlyStopping(monitor="val_loss", patience=3) # Stop training if the validation loss does not improve for 3 epochs
]


# Define the trainer
trainer = pl.Trainer(
        max_epochs=5,
        devices=1,
        callbacks=checkpoint_callback,
        strategy= "auto" if test_on == "cpu" else "ddp_notebook" ,
        accelerator= "cpu" if test_on == "cpu" else "gpu",
        log_every_n_steps=10,
        logger=csv_logger
    )


## Train the minimalistic model

In [ ]:
trainer.fit(mini_Model,train_dataloaders= train_loader, val_dataloaders= val_loader)


In [ ]:
raise ValueError("We did not implement a test function for the mini introduction :)")
trainer.test(mini_Model, test_dataloaders=test_loader)